# 00: Connect SQL to the deployed Foundry resource

Create a database master key and managed-identity credential so `Healthcare` can connect to the deployed Foundry embedding model.

Open the repository in VS Code, select the **MSSQL** kernel, and connect to the assigned SQL server with Microsoft Entra authentication. Select **Healthcare**, not `master`. Run one SQL cell at a time.

Your Entra account connects to SQL; the SQL server's managed identity calls Foundry. No API key is needed.


## 1. Verify the database

Verify that you are connected to `Healthcare` and that all 207 synthetic cases are available. Contact the facilitator if this check fails.


In [ ]:
SET NOCOUNT ON;
IF DB_NAME() <> N'Healthcare' OR CONVERT(INT, SERVERPROPERTY('EngineEdition')) <> 5
    THROW 51000, 'Connect to the dedicated Healthcare database on Azure SQL Database.', 1;
IF HAS_PERMS_BY_NAME(DB_NAME(), N'DATABASE', N'CONTROL') <> 1
    THROW 51000, 'Connect with the assigned lab Entra administrator to configure database credentials.', 1;
IF OBJECT_ID(N'dbo.Cases', N'U') IS NULL
    THROW 51000, 'The host must prepare dbo.Cases and the 207 synthetic cases before this notebook.', 1;
DECLARE @Count INT;
EXEC sys.sp_executesql N'SELECT @Rows = COUNT(*) FROM dbo.Cases;', N'@Rows INT OUTPUT', @Rows = @Count OUTPUT;
IF @Count <> 207
    THROW 51000, 'Expected 207 synthetic cases. Contact the facilitator; do not reset the database.', 1;
SELECT DB_NAME() AS WorkshopDatabase, @Count AS SeededCases;


## 2. Create the database master key

The master key protects database credentials. This cell preserves an existing key.


In [ ]:
IF DB_NAME() <> N'Healthcare' OR CONVERT(INT, SERVERPROPERTY('EngineEdition')) <> 5
    THROW 51000, 'Connect to the dedicated Healthcare database on Azure SQL Database.', 1;
IF NOT EXISTS (SELECT 1 FROM sys.symmetric_keys WHERE name = N'##MS_DatabaseMasterKey##')
    CREATE MASTER KEY;
SELECT name, algorithm_desc FROM sys.symmetric_keys WHERE name = N'##MS_DatabaseMasterKey##';


## 3. Save your Foundry endpoint in the database

1. Open [Resource groups in Azure Portal](https://portal.azure.com/#browse/resourcegroups) and sign in with your assigned lab account.
2. Select the resource group assigned to your lab. In its resource list, select the Foundry resource named **`foundry-<lab-id>-cu`**, not the SQL server or the `healthcare` project.
3. In the Foundry resource's left menu, expand **Resource Management** and select **Keys and Endpoint**. Use the menu's search box to find **Keys and Endpoint** if needed.
4. Copy the resource endpoint with the format **`https://foundry-<lab-id>-cu.services.ai.azure.com`**. The resource's **Overview** page may also show its endpoints. If multiple endpoints are listed, choose the one ending in **`.services.ai.azure.com`**. If that address is not listed, ask the facilitator for the resource-level Foundry endpoint.
5. Return to this notebook and replace the quoted URL assigned to **`@Endpoint`** in the next SQL cell. Keep the `N'...'` SQL string syntax, then run the cell.

Use the resource endpoint, not the portal address, a project URL ending `/api/projects/healthcare`, or a full embeddings request URL.

The credential saves the endpoint for later notebooks. Matching existing configuration is reused.


In [ ]:
-- STUDENT INPUT: Paste the actual resource-level Foundry endpoint.
DECLARE @Endpoint NVARCHAR(4000) = N'https://YOUR-FOUNDRY-RESOURCE.services.ai.azure.com';

SET NOCOUNT ON;
IF DB_NAME() <> N'Healthcare' OR CONVERT(INT, SERVERPROPERTY('EngineEdition')) <> 5
    THROW 51000, 'Connect to the dedicated Healthcare database on Azure SQL Database.', 1;
SET @Endpoint = TRIM(@Endpoint);
IF RIGHT(@Endpoint, 1) = N'/' SET @Endpoint = LEFT(@Endpoint, LEN(@Endpoint) - 1);
IF @Endpoint IS NULL
    OR @Endpoint NOT LIKE N'https://foundry-%-cu.services.ai.azure.com'
    OR LEN(@Endpoint) > 128
    OR @Endpoint COLLATE Latin1_General_100_BIN2 LIKE N'%[^-a-zA-Z0-9:/.]%'
    OR CHARINDEX(N'/', @Endpoint, 9) > 0
    OR CHARINDEX(N':', @Endpoint, 7) > 0
    THROW 51000, 'Paste the actual resource-level Foundry endpoint for this lab, not a project or request URL.', 1;
IF NOT EXISTS (SELECT 1 FROM sys.symmetric_keys WHERE name = N'##MS_DatabaseMasterKey##')
    THROW 51000, 'Complete Step 2 to create the database master key.', 1;
IF EXISTS (SELECT 1 FROM sys.database_scoped_credentials WHERE name = @Endpoint AND credential_identity <> N'Managed Identity')
    THROW 51000, 'This endpoint credential uses a different identity. Ask the facilitator to review its dependencies.', 1;
IF EXISTS (SELECT 1 FROM sys.database_scoped_credentials
    WHERE credential_identity = N'Managed Identity'
      AND name LIKE N'https://foundry-%-cu.services.ai.azure.com' AND name <> @Endpoint)
    THROW 51000, 'A different workshop Foundry credential already exists. Ask the facilitator to review it; do not replace it.', 1;
IF NOT EXISTS (SELECT 1 FROM sys.database_scoped_credentials WHERE name = @Endpoint)
BEGIN
    DECLARE @SQL NVARCHAR(MAX) = N'CREATE DATABASE SCOPED CREDENTIAL ' + QUOTENAME(@Endpoint)
        + N' WITH IDENTITY = ''Managed Identity'', SECRET = ''{"resourceid":"https://cognitiveservices.azure.com"}'';';
    EXEC sys.sp_executesql @SQL;
END;
SELECT name AS FoundryEndpoint, credential_identity FROM sys.database_scoped_credentials WHERE name = @Endpoint;


## 4. Continue to notebook 01

The result above should show your Foundry resource URL and **Managed Identity**.

Open [01: Beyond Keywords](01-beyond-keywords.ipynb) and run its Step 00. All three checks must say **Yes**: Workshop database, Seeded cases, and Prepared Foundry credential.

Notebook 01 registers the external embedding model using the saved endpoint. Notebook 02 makes the first live embedding request.

**Learn more:** [Database-scoped credentials](https://learn.microsoft.com/en-us/sql/t-sql/statements/create-database-scoped-credential-transact-sql?view=sql-server-ver17).

[Attendee guide](README.md) | Next: [Notebook 01](01-beyond-keywords.ipynb)
